# 🔐 **Security Evaluation of a Face Recognition System**

## Baseline Model Evaluation on NN1 (Inception ResNet V1)

**Academic Year:** 2024-2025  
**Group:** 04

---

### 👥 Team Members
- **Agostino Cardamone** — `0622702276`
- **Asja Antonucci**     — `0622702437`
- **Chiara Ferraioli**   — `0622702169`

---

### 📚 Overview of This Section

1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Dataset Setup](#2-dataset-setup)
3. [Dataset Overview](#3-dataset-overview)
4. [Baseline Model Evaluation](#4-baseline-model-evaluation)

## 1. Setup and Data Loading

#### Environment Setup

To ensure reproducibility and avoid package conflicts, it is strongly recommended to run all experiments in an isolated environment. We use Conda to create and manage the project environment, and all Python dependencies are listed in the requirements.txt file.

In [ ]:
# 1) Create a new environment named “aic_env”
#conda create -n aic_env python=3.10 -y

# 2) Switch into the new environment
# On Windows:
# conda activate aic_env
# On Linux/macOS:
# source activate aic_env

# 3) Install all dependencies
#!pip install -r requirements.txt

import os 

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print("torch.version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

#### Dataset Configuration and Paths

This section defines the core paths and settings used throughout the project to manage the dataset structure and preprocessing behavior.

- `dataset_dir` points to the root directory containing the dataset files.
- `dataset_selection` controls whether to regenerate the test set from scratch (⚠️set to True only if you have extracted `vggface2_train` inside `dataset/vggface2_train/trainset`⚠️)
- `mtcnn_processing_nn1` determines whether to apply MTCNN face alignment for NN1 preprocessing.
- `test_set_rnd` specifies whether the test set should be built randomly or from the pre-defined class list in `test_set.csv`.

Metadata and folder structure:
- `vgg2_dataset_annotations_path` points to `identity_meta.csv`, which contains class metadata (name, gender, etc.).
- `test_set_data_folder` is the location of test samples.
- `test_set_annotations_folder` contains the CSV file describing the test set structure.

All experiment outputs will be saved to:
- `results_folder` — for accuracy results and evaluations.
- `adversarial_folder` — for storing generated adversarial images.

The variable `device` automatically selects GPU if available, otherwise defaults to CPU.

In [ ]:
from utils import *         # Project-specific utilities and imports

# —————————————————————————————————————————————
#           Paths and configuration
# —————————————————————————————————————————————

# Base directory containing all dataset-related files
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# If True, a new test set will be built by sampling and copying images from the original VGGFace2 dataset
# NOTE: This requires the dataset to be downloaded and extracted under 'vggface2_train/trainset'
# If False, the existing CSV files will be loaded without modifying or copying any images
dataset_selection = False  

# If True, test images will be aligned and cropped using MTCNN preprocessing (for NN1 compatibility)
mtcnn_processing_nn1 = False

# If True, the test set will be built via random sampling of identities and images
# If False, the test set will be built based on predefined class IDs listed in 'test_set.csv'
test_set_rnd = False

# Path to VGGFace2 identity metadata (includes Class_ID, Name, Gender, etc.)
vgg2_dataset_annotations_path = os.path.join(dataset_dir, 'identity_meta.csv')

# Folder structure for test set files and images
test_set_folder              = os.path.join(dataset_dir, 'testset')
test_set_data_folder         = os.path.join(test_set_folder, 'samples')
test_set_annotations_folder  = os.path.join(test_set_folder, 'test_set.csv')

# Directory to store evaluation results (e.g., SEC curves)
results_folder = os.path.join(os.getcwd(), 'results')

# Directory to save generated adversarial examples
adversarial_folder = os.path.join(os.getcwd(), 'attacks')

# Device configuration: use GPU if available, otherwise fallback to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 2. Dataset Setup

#### Test Set Creation (Optional)

This block handles the optional creation of a new test set by sampling 100 identities from the full VGGFace2 training dataset.

- First, the metadata CSV is loaded (`identity_meta.csv`), which contains gender, name, and class ID for all identities.
- If `dataset_selection` is set to `True`, the script:
  - Randomly samples up to 50 male and 50 female identities, ensuring each class has at least 10 images.
  - Falls back to additional sampling if fewer than 100 valid identities are found.
  - Copies the selected identities (10 images each) into a local `testset/samples/` folder.
  - Saves the selected subset into `test_set.csv`.

If `dataset_selection` is `False`, the script simply loads an existing test set from `test_set.csv`.

Finally, a maximum of 100 class folders are copied to the test directory. Only classes with ≥10 images are retained, and a clean CSV is regenerated with the successfully copied identities.

In [ ]:
# Read the full VGGFace2 metadata CSV into a DataFrame (contains 9131 classes)
vgg2_dataset = pd.read_csv(
    filepath_or_buffer=vgg2_dataset_annotations_path,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

# ————————————————————————————————————————————————————————————————
#   (Optional) Build a fresh test‐set: sample + copy image files
# ————————————————————————————————————————————————————————————————

# Check if the dataset is already created
if dataset_selection:
    # Split the full VGG2 metadata into male and female subsets
    male_dataset   = vgg2_dataset[vgg2_dataset['Gender'] == 'm']
    female_dataset = vgg2_dataset[vgg2_dataset['Gender'] == 'f']

    # Ensure the test-set folder and the `samples/` subfolder exist
    if not os.path.exists(test_set_folder):
        os.makedirs(test_set_folder)
        
    if not os.path.exists(test_set_data_folder):
        os.makedirs(test_set_data_folder)

    # Source root for the full VGGFace2 train images on your machine
    train_root = (
        "C:/Users/chiar/Desktop/Uni/AI for Cybersecurity/AI-for-Cybersecurity-Project/dataset/vggface2_train/train"  # e.g., "/home/user/datasets/vggface2_train/trainset"
    )

    # Stores all the sampled identity rows that are successfully copied
    copied_rows = []

    # Sampling helper: adds classes only if folder exists and has ≥10 images
    def try_add_samples(df, max_count, current_count, selected_ids):
        added = 0
        result = []
        for _, row in df.iterrows():
            if added >= max_count - current_count:
                break
            class_id = row['Class_ID']
            
            src_dir = os.path.join(train_root, class_id)

            if not os.path.exists(src_dir):
                continue
            all_images = os.listdir(src_dir)
            if len(all_images) < 10:
                continue
            if class_id in selected_ids:
                continue

            selected_ids.add(class_id)
            result.append(row)
            added += 1
        return result

    # Randomly sample or load fixed set
    if test_set_rnd:
        selected_ids = set()

        # Step 1: sample up to 50 male classes
        male_sample = try_add_samples(male_dataset.sample(frac=1, random_state=42), max_count=50, current_count=0, selected_ids=selected_ids)

        # Step 2: sample up to 50 female classes
        female_sample = try_add_samples(female_dataset.sample(frac=1, random_state=43), max_count=50, current_count=0, selected_ids=selected_ids)

        # Combine valid samples
        total_samples = male_sample + female_sample

        # Step 3: if still under 100 identities, fill with extras
        if len(total_samples) < 100:
            missing = 100 - len(total_samples)

            extra_females = try_add_samples(female_dataset.sample(frac=1, random_state=44), max_count=missing, current_count=0, selected_ids=selected_ids)
            extra_males   = try_add_samples(male_dataset.sample(frac=1, random_state=45), max_count=missing, current_count=0, selected_ids=selected_ids)

            if len(extra_females) >= missing:
                total_samples += extra_females[:missing]
            elif len(extra_males) >= missing:
                total_samples += extra_males[:missing]
            else:
                total_samples += (extra_females + extra_males)[:missing]

        # Save ordered test set to CSV
        test_set = pd.DataFrame(total_samples).sort_values(by="Name").reset_index(drop=True)
        test_set.to_csv(test_set_annotations_folder, index=False)

    else:
        # Load the predefined test set from CSV
        test_set = pd.read_csv(
            filepath_or_buffer=test_set_annotations_folder,
            sep=',',
            skipinitialspace=True,
            engine='python'
        )

    # Copy up to 100 identity folders, each with exactly 10 images
    classes_copied = 0
    for _, row in test_set.iterrows():
        if classes_copied >= 100:
            break

        class_id   = row['Class_ID']
        class_name = row['Name']

        src_dir = os.path.join(train_root, class_id)
        dst_dir = os.path.join(test_set_data_folder, class_name)

        # Verifica se la cartella sorgente esiste prima di procedere
        if not os.path.exists(src_dir):
            print(f"Skipping {class_id}/{class_name} → source folder not found")
            continue

        try:
            os.makedirs(dst_dir, exist_ok=True)
            all_images = os.listdir(src_dir)

            if len(all_images) < 10:
                print(f"Skipping {class_id}/{class_name} → not enough images")
                continue

            for img_filename in all_images[:10]:
                src_path = os.path.join(src_dir, img_filename)
                dst_path = os.path.join(dst_dir, img_filename)
                shutil.copy(src_path, dst_path)
            
            copied_rows.append(row)  
            classes_copied += 1

        except Exception as e:
            print(f"Error: could not copy for {class_id}/{class_name} → {e}")
    
    test_set_copied = pd.DataFrame(copied_rows)
    test_set_copied = test_set_copied.sort_values(by="Name").reset_index(drop=True)
    test_set_copied.to_csv(test_set_annotations_folder, index=False)


## 3. Dataset Overview

#### Load Test Set and Class Labels

This block performs two critical initializations:

1. **Load test set metadata**
   - The file `test_set.csv` is read into a DataFrame.
   - It contains exactly 100 test identities, each with 10 face images located in `testset/samples/`.
   - The total number of expected images is computed as `100 × 10 = 1000`.

2. **Load class label mappings**
   - The face recognition model requires access to the full list of class names (8631 identities).
   - These are loaded from a `.npy` file originally published by the official [`rcmalli/keras-vggface`](https://github.com/rcmalli/keras-vggface) repository.
   - If the file is not present locally, it is automatically downloaded.
   - Any surrounding whitespace in label strings is stripped to ensure clean formatting.

This step is necessary to convert model outputs (class indices) into readable identity labels.

In [ ]:
# —————————————————————————————————————————————
#              Load annotation CSVs
# —————————————————————————————————————————————

# Read the existing test_set.csv describing our 100 test identities
# (each identity will have 10 samples in the folder structure)
test_set = pd.read_csv(
    filepath_or_buffer=test_set_annotations_folder,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

# Calculate how many total images we expect in the test set:
# number of identities × 10 images each
test_set_size = len(test_set) * 10

# ————————————————————————————————————————————
#  Download and load the class‐label mapping
# ————————————————————————————————————————————

# Our face‐recognition model expects a NumPy array of all 8631 labels.
# We download it from the official rcmalli/keras-vggface repo if missing.
labels_url = (
    "https://github.com/rcmalli/keras-vggface/"
    "releases/download/v2.0/rcmalli_vggface_labels_v2.npy"
)
labels_path = os.path.join(dataset_dir, 'rcmalli_vggface_labels_v2.npy')

# Ensure the directory for labels_path exists
os.makedirs(os.path.dirname(labels_path), exist_ok=True)

# Download only if the file does not already exist on disk
if not os.path.exists(labels_path):
    print(f"Downloading LABELS to {labels_path}…")
    urllib.request.urlretrieve(labels_url, labels_path)

# Load the .npy file into a NumPy array and strip any padding whitespace
LABELS = np.load(labels_path)
LABELS = np.char.strip(LABELS)

## 4. Baseline Model Evaluation

#### Load and Prepare Face Recognition Model (NN1)

This section sets up the face recognition model referred to as **NN1**, based on the `InceptionResnetV1` architecture provided by the `facenet-pytorch` library.

- The model is initialised with weights pre-trained on the **VGGFace2** dataset, which contains over 8,000 people identities.
- It is set to evaluation mode (`.eval()`), disabling stochastic layers such as dropout and batch normalisation updates, ensuring deterministic inference.
- By default, the model outputs a 512-dimensional feature embedding for each face. However, enabling the `.classify = True` flag appends a classification head, allowing the model to directly output class logits over the **8,631 identities** present in the VGGFace2 training set.

In [ ]:
from facenet_pytorch import InceptionResnetV1

# ——————————————————————————————————————————————————————
#   Initialize the pre-trained face-recognition model
# ——————————————————————————————————————————————————————

# We use the InceptionResnetV1 architecture from the facenet-pytorch package,
# pre-trained on the VGGFace2 dataset for high-quality face embeddings.
# By calling .eval(), we set the model to inference mode (disables dropout, batchnorm updates).
# We then move the model to the appropriate device (GPU if available, else CPU).
nn1 = InceptionResnetV1(
    pretrained='vggface2'  # load weights trained on the VGGFace2 face dataset
).eval().to(device)         # switch to evaluation mode and transfer to GPU/CPU

# —————————————————————————————————————————————
#           Enable classification head
# —————————————————————————————————————————————

# By default, InceptionResnetV1 returns 512-dimensional embeddings.
# Setting .classify instructs the model to append a linear classification
# layer on top of the embeddings, so that nn1(input) returns raw class logits
# for all identities in VGGFace2 (8 631 classes), instead of embeddings.
nn1.classify = True

#### Face Detection and Alignment (MTCNN)

To ensure consistent and high-quality input for face recognition, the **MTCNN** (Multi-task Cascaded Convolutional Networks) model is used as a preprocessing stage. MTCNN carries out multiple tasks in sequence to detect and align faces with a high degree of accuracy. Specifically, it:

- Locates the most prominent face within each image.
- Aligns facial features based on detected landmarks (e.g., eyes, nose, and mouth).
- Crops and resizes the face region to a fixed resolution of **160×160 pixels**, matching the expected input size of the recognition model.

The aligned faces are returned as PyTorch tensors, ready for immediate use in model inference. Additionally, these preprocessed images can be optionally saved to disk, allowing the system to bypass real-time face detection in subsequent runs — a significant benefit in terms of efficiency.

> When the flag `mtcnn_processing_nn1` is set to `True`, the aligned face crops are automatically stored in the directory:  
> `testset/cropped_faces_nn1/{class_name}/img_XXX.jpg`

In [ ]:
# ———————————————————————————————————————————————————
#  Initialize the face detector and aligner (MTCNN)
# ———————————————————————————————————————————————————
# We use MTCNN from facenet-pytorch to detect, crop, and align faces in one step.
# When you call face_detector_nn1(img_batch), it returns a tensor of shape [B, 3, image_size, image_size]
# containing the aligned face crops, ready to feed into nn1 or an adversarial attack.

face_detector_nn1 = MTCNN(
    image_size=160,                             # int: output height/width of each face crop (default=160)
    margin=0,                                   # int: number of pixels to expand the face bounding box (default=0)
    min_face_size=20,                           # int: minimum face size (in pixels) that the detector will attempt to locate (default=20)
    thresholds=[0.6, 0.7, 0.7],                 # list of 3 floats: score thresholds for each detection stage—
                                                #   P-Net, R-Net, and O-Net respectively (default=[0.6, 0.7, 0.7])
    factor=0.709,                               # float: scale factor between pyramid levels; controls the search granularity (default=0.709)
    post_process=True,                          # bool: whether to apply face alignment post-processing (True)
    select_largest=True,                        # bool: if multiple faces are detected, return only the largest one (True)
    selection_method="center_weighted_size",    # str: heuristic for choosing among multiple detections—
                                                #   options include "largest" or "center_weighted_size" (default="center_weighted_size")
    keep_all=False,                             # bool: if True, return all detected faces; if False, return only one (default=False)
    device=device                               # torch.device or str: computation device, e.g. "cuda:0" or "cpu"
)


In this section, we prepare the test set so it can be efficiently used during inference. The aim is to create a `DataLoader` that iterates over face images, optionally applying preprocessing steps, and associates each image with the correct identity label.

The process includes the following components:

- **Image Transformations**  
  A basic image preprocessing pipeline is defined using `torchvision.transforms`. If MTCNN alignment is not enabled, each image is resized to 160×160 pixels (the input size required by the face recognition model) and then converted to a tensor.

- **Dataset Definition via ImageFolder**  
  The test images are loaded using `ImageFolder`, which expects the directory structure to be organised such that each identity has its own folder. The dataset automatically assigns a numerical label to each subfolder and loads all images accordingly.

- **Label Mapping**  
  A custom mapping `idx_to_class` is constructed to associate the internal numeric labels used by `ImageFolder` with the actual identity names provided in the test set metadata. This ensures label consistency throughout the evaluation process.

- **Final DataLoader**  
  The dataset is wrapped in a `DataLoader` for iteration. No parallel workers (`num_workers=0`) are used to maintain compatibility and simplicity. This `DataLoader` can now be used to either:
  - Apply face detection and alignment via MTCNN (if enabled), or
  - Pass pre-aligned images directly to the recognition model.

This structure ensures that the test set is handled in a clean, modular, and reproducible way.

In [ ]:
# Define the transforms for the DataLoader
transforms = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# Create a DataLoader for the test set by using the ImageFolder dataset
dataset = datasets.ImageFolder(root=test_set_data_folder, transform=transforms if not mtcnn_processing_nn1 else None)
dataset.idx_to_class = {i: c.replace('', '') for i, c in enumerate(test_set['Name'])}
dataloader = DataLoader(dataset, collate_fn=collate_fn, num_workers=0)                                                                  

In this section, we perform face alignment on the test set using **MTCNN** (Multi-task Cascaded Convolutional Networks) and construct a clean, reusable dataset ready for evaluation. This step is executed only if the configuration flag `mtcnn_processing_nn1` is set to `True`.

To avoid repeating the alignment procedure on every run, the script first checks if the directory `testset/cropped_faces_nn1/` exists and contains files. If it does, the system assumes that the aligned dataset has already been generated and logs a message to skip the alignment step.

This caching strategy ensures **efficiency and reproducibility** across experiments.

If alignment is required, the script iterates over each image in the `DataLoader` and applies the following logic:

- For each image, it retrieves the associated identity label and ensures that a corresponding subfolder exists.
- It keeps track of how many aligned samples have been saved for that identity using a simple counter.
- It defines a target filename (e.g., `img_000.jpg`, `img_001.jpg`) for the aligned output.
- The image is passed to MTCNN, which detects and aligns the most prominent face.
- If a face is detected, the aligned image is:
  - Saved to the appropriate folder on disk.
  - Appended to a list of tensors (`x_test_aligned_nn1`).
  - Paired with its class label (`y_test_aligned_nn1`).
- If no face is found, the script logs a warning and skips the image.

This ensures that each identity in the test set has a fixed number of high-quality, aligned face crops, improving model robustness and evaluation stability.

Once all images have been processed:

- The list of aligned images is stacked into a single tensor.
- The list of labels is converted into a tensor of class indices.
- Both tensors are saved to disk as:
  - `x_test_aligned_nn1.pt` — aligned image tensors.
  - `y_test_aligned_nn1.pt` — corresponding labels.

Additionally, a `DataLoader` is constructed using `TensorDataset`, bundling images and labels into a ready-to-use object. This `DataLoader`, along with the `idx_to_class` mapping, is saved as `dataloader_aligned_nn1.pt` for future reuse.

In [ ]:
if mtcnn_processing_nn1:
    cropped_root = os.path.join(test_set_folder, 'cropped_faces_nn1')
    os.makedirs(cropped_root, exist_ok=True)

    if os.path.isdir(cropped_root) and os.listdir(cropped_root):
        logger.info(f"Found existing crops in {cropped_root}, skipping face cropping.")
    else:
        x_test_aligned_nn1 = []
        y_test_aligned_nn1 = []
        print_done = False
        class_counters = dict()

        for i, (img, y) in enumerate(dataloader):
            class_name = dataset.idx_to_class[y].strip()
            class_dir = os.path.join(cropped_root, class_name)
            os.makedirs(class_dir, exist_ok=True)

            if class_name not in class_counters:
                class_counters[class_name] = 0
            count = class_counters[class_name]
            class_counters[class_name] += 1
                
            save_path = os.path.join(class_dir, f'img_{count:03d}.jpg')
            
            x_aligned = face_detector_nn1(img, return_prob=False, save_path=save_path)

            if x_aligned is not None:
                x_test_aligned_nn1.append(x_aligned)
                y_test_aligned_nn1.append(y)

            else:
                logger.info(f'Face not detected in image {i}')

        x_test_aligned_nn1 = torch.stack(x_test_aligned_nn1)
        y_test_aligned_nn1 = torch.tensor(y_test_aligned_nn1)

        torch.save(x_test_aligned_nn1, 'dataset/x_test_aligned_nn1.pt')
        torch.save(y_test_aligned_nn1, 'dataset/y_test_aligned_nn1.pt')

        dataloader_aligned_nn1 = DataLoader(TensorDataset(x_test_aligned_nn1, y_test_aligned_nn1), collate_fn=collate_fn)
        dataloader_aligned_nn1.dataset.idx_to_class = dataset.idx_to_class

        torch.save(dataloader_aligned_nn1, 'dataset/dataloader_aligned_nn1.pt')

Once the aligned dataset has been created and saved, we can efficiently reload it and prepare it for inference or evaluation without repeating the alignment process. 

The aligned dataset, which was stored in a PyTorch `DataLoader`, includes both the face images and their corresponding identity labels. Once loaded, the individual samples are unpacked and converted into two tensors: one containing all the aligned images, and the other containing the associated class labels.

To ensure compatibility with downstream processing steps, the image tensor is converted from PyTorch format to a NumPy array. This is especially useful when the data needs to be passed into frameworks such as **ART (Adversarial Robustness Toolbox)**, which often expect NumPy inputs for generating adversarial examples.

Finally, the script reconstructs a reverse label mapping that allows us to convert class names (i.e., the folder names used in `ImageFolder`) back into numerical indices. This is essential for consistent evaluation and interpretation of results, especially when comparing predicted and true labels.


In [ ]:
# Load the aligned DataLoader
dataloader_aligned_nn1 = torch.load('dataset/dataloader_aligned_nn1.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
x_test_aligned_nn1, y_test_aligned_nn1 = zip(*[(sample[0], sample[1]) for sample in dataloader_aligned_nn1])

# Create the batches of the tensors of the images and labels
x_test_aligned_nn1 = torch.stack(x_test_aligned_nn1)
y_test_aligned_nn1 = torch.tensor(y_test_aligned_nn1)

x_test_aligned_nn1 = x_test_aligned_nn1.cpu().numpy() 

class_to_idx = {v: k for k, v in dataset.idx_to_class.items()}

#### Evaluation of NN1 on the Clean Test Set

With the face recognition model (NN1) initialised and the test set properly aligned, we now proceed to evaluate the model's classification performance. This step establish the model's accuracy under clean (i.e., non-adversarial) conditions, and it serves as the baseline for all further robustness analysis.

The model is evaluated using a custom utility function called `evaluate_model(...)`, which has been implemented to streamline the prediction process. This function:

- Sets the model to **evaluation mode**, ensuring that layers such as dropout and batch normalisation behave deterministically.
- Iterates through the test set using the aligned `DataLoader`.
- For each input image, obtains the model’s predicted class index and maps it to the corresponding class name using the `LABELS` array.
- Retrieves the ground truth label from the dataset and maps it to a string using `idx_to_class`.

This process produces two parallel lists:
- `y_true`: the ground truth identity names for each test image.
- `y_pred`: the predicted identity names returned by the model.

By using human-readable labels instead of raw indices, the results become more interpretable and suitable for further analysis (e.g., accuracy computation, error case review).

Once inference is complete, both the predicted and true labels are saved to disk as `.pt` files. These files can later be loaded for performance evaluation, statistical reporting, or visualisation.

In [ ]:
# Get the embeddings for the test set
y_true, y_pred_nn1 = evaluate_model(nn1, dataloader_aligned_nn1, LABELS)

# Save the embeddings for the test set
torch.save(y_true, 'y_true.pt')
torch.save(y_pred_nn1, 'y_pred_nn1.pt')

In this phase, we evaluate the face recognition model **NN1** on the clean (unaltered) version of the test set. This step serves to establish a reliable baseline for model performance in the absence of adversarial perturbations.

To quantify the model's performance:

- The `print_basic_metrics` function is used to display:
  - The total number of samples evaluated.
  - The number of correctly and incorrectly classified examples.

- The `accuracy_score` function from `sklearn` is then used to calculate the overall classification accuracy of NN1 on the clean dataset. The result is printed as a percentage for clarity.

This clean accuracy represents the model’s baseline performance without adversarial interference.

To gain insight into where the model struggles, the script analyses misclassifications:

- It pairs each incorrect prediction with its corresponding ground truth.
- Using Python’s `Counter`, it identifies the most frequent misclassification patterns (e.g., person A often mistaken for person B).
- This helps in spotting recurring confusions that may indicate class overlap, poor data quality, or overfitting on certain identities.

The function `plot_predicted_images` is used to generate a visual summary of the model’s predictions on a subset of the test set:

- For each selected identity, one sample image is displayed.
- Above each image, the true identity is shown.
- Below each image, the predicted label is displayed:
  - In **green** if the prediction is correct.
  - In **red** if the prediction is incorrect.

This visual tool provides intuitive insight into the model’s successes and failures and is especially valuable for identifying qualitative issues that are not visible through metrics alone.

In [ ]:
clean_acc_nn1_folder = os.path.join(results_folder, 'clean_acc_nn1')
os.makedirs(clean_acc_nn1_folder, exist_ok=True)
img_file_path = os.path.join(clean_acc_nn1_folder, 'nn1_clean_predictions.png')

y_true = torch.load('y_true.pt')
y_pred_nn1 = torch.load('y_pred_nn1.pt')

selected_classes = ['Andrea_Bocelli','Gigi_DAlessio','Diego_Abatantuono','Diego_Maradona','Francesco_Totti','Dries_Mertens']

idx_test_images = []
for name in selected_classes:
    if name in class_to_idx:
        idx_test_images.append(class_to_idx[name])
    else:
        print(f"Name not found: {name}")

correct, incorrect = print_basic_metrics(y_true, y_pred_nn1)
clean_acc_nn1 = accuracy_score(y_true, y_pred_nn1)
print(f"NN1 clean accuracy: {clean_acc_nn1*100:.2f}%")

mismatches = [(t, p) for t, p in zip(y_true, y_pred_nn1) if t != p]
print("\nNot Correctly classified:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"❌ {t} → {p}  ({count} volte)")

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    y_true=y_true,
    y_pred=y_pred_nn1,
    id_test_images=idx_test_images,
    image_idx=0,
    save_path=img_file_path
)